# Perkenalan Dataset

Notebook ini menggunakan dataset prediksi kelulusan mahasiswa. Dataset berisi informasi akademik dan non-akademik mahasiswa seperti nilai IPS per semester, jumlah SKS, kehadiran, aktivitas organisasi, status beasiswa, asal sekolah, dan penghasilan orang tua.

Target prediksi pada dataset ini adalah `status_kelulusan`, yaitu kategori status kelulusan mahasiswa. Notebook eksperimen ini hanya mencakup tahap pengenalan dataset, import library, pemuatan dataset, exploratory data analysis, dan data preprocessing sampai data siap dilatih tersimpan di folder `preprocessing/`.


# Import Library


In [1]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Library berhasil di-import")


Library berhasil di-import


# Memuat Dataset


In [2]:
DATA_PATH = Path("namadataset_raw/student_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print("Daftar kolom:")
print(df.columns.tolist())
df.head()


Dataset shape: (1000, 16)
Daftar kolom:
['ips_1', 'ips_2', 'ips_3', 'ips_4', 'ips_5', 'ips_6', 'ips_7', 'jumlah_sks', 'umur', 'jenis_kelamin', 'asal_sekolah', 'organisasi', 'beasiswa', 'jumlah_tanggungan', 'penghasilan_ortu', 'status_kelulusan']


,ips_1,ips_2,ips_3,ips_4,ips_5,ips_6,ips_7,jumlah_sks,umur,jenis_kelamin,asal_sekolah,organisasi,beasiswa,jumlah_tanggungan,penghasilan_ortu,status_kelulusan
0,2.12,1.56,1.79,3.02,2.72,2.18,2.94,150,24.0,Perempuan,SMK,Ya,Tidak,3,5-10jt,Drop Out
1,3.85,2.63,1.74,3.39,3.42,2.42,1.52,116,28.0,Laki-laki,SMK,Ya,Ya,4,>10jt,Lulus Terlambat
2,3.20,3.62,3.72,1.75,3.28,3.56,3.62,86,21.0,Laki-laki,SMA,Tidak,Ya,2,2-5jt,Lulus Terlambat
3,2.80,3.20,1.75,2.87,1.46,2.02,2.84,134,18.0,Perempuan,SMA,Tidak,Tidak,5,>10jt,Lulus Terlambat
4,1.47,3.42,1.82,2.72,1.45,3.61,1.47,64,25.0,Laki-laki,SMK,Ya,Tidak,2,2-5jt,Drop Out


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ips_1              1000 non-null   float64
 1   ips_2              1000 non-null   float64
 2   ips_3              980 non-null    float64
 3   ips_4              1000 non-null   float64
 4   ips_5              1000 non-null   float64
 5   ips_6              1000 non-null   float64
 6   ips_7              1000 non-null   float64
 7   jumlah_sks         1000 non-null   int64  
 8   umur               985 non-null    float64
 9   jenis_kelamin      1000 non-null   object 
 10  asal_sekolah       1000 non-null   object 
 11  organisasi         1000 non-null   object 
 12  beasiswa           1000 non-null   object 
 13  jumlah_tanggungan  1000 non-null   int64  
 14  penghasilan_ortu   985 non-null    object 
 15  status_kelulusan   1000 non-null   object 
dtypes: float64(8), int64(2), 

In [4]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ips_1,1000.0,NaN,NaN,NaN,2.47085,0.876484,1.01,1.7075,2.49,3.2325,4.0
ips_2,1000.0,NaN,NaN,NaN,2.52098,0.876516,1.01,1.72,2.56,3.28,4.0
ips_3,980.0,NaN,NaN,NaN,2.503255,0.872639,1.0,1.78,2.5,3.28,3.99
ips_4,1000.0,NaN,NaN,NaN,2.4711,0.859313,1.0,1.7275,2.45,3.21,4.0
ips_5,1000.0,NaN,NaN,NaN,2.48254,0.860619,1.0,1.7375,2.485,3.22,3.99
ips_6,1000.0,NaN,NaN,NaN,2.4954,0.867582,1.02,1.7475,2.47,3.22,4.0
ips_7,1000.0,NaN,NaN,NaN,2.48006,0.86917,1.0,1.72,2.455,3.27,4.0
jumlah_sks,1000.0,NaN,NaN,NaN,110.711,28.963397,60.0,85.0,112.0,137.0,159.0
umur,985.0,NaN,NaN,NaN,23.504569,3.436448,18.0,20.0,24.0,27.0,29.0
jenis_kelamin,1000,2,Laki-laki,512,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Exploratory Data Analysis (EDA)


In [5]:
print("Distribusi target status_kelulusan:")
print(df["status_kelulusan"].value_counts())

plt.figure(figsize=(8, 5))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
df["status_kelulusan"].value_counts().plot(kind="bar", color=colors, edgecolor="black")
plt.title("Distribusi Status Kelulusan")
plt.xlabel("Status Kelulusan")
plt.ylabel("Jumlah")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "eda_target_distribution.png", dpi=150)
plt.show()


Distribusi target status_kelulusan:
status_kelulusan
Lulus Terlambat      525
Drop Out             263
Lulus Tepat Waktu    212
Name: count, dtype: int64


In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_percentage": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_count", ascending=False)

print("Ringkasan missing values:")
print(missing_df if not missing_df.empty else pd.DataFrame({"info": ["Tidak ada missing values"]}))

if not missing_df.empty:
    plt.figure(figsize=(8, 4))
    missing_df["missing_count"].plot(kind="barh", color="#e74c3c")
    plt.title("Missing Values per Kolom")
    plt.xlabel("Jumlah Missing")
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "eda_missing_values.png", dpi=150)
    plt.show()


Ringkasan missing values:
                  missing_count  missing_percentage
ips_3                        20                 2.0
umur                         15                 1.5
penghasilan_ortu             15                 1.5


In [7]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Kolom numerik: {numeric_cols}")

n_cols = 4
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, max(4, 3 * n_rows)))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(numeric_cols):
    df[col].hist(bins=20, ax=axes[i], color="#3498db", edgecolor="black", alpha=0.75)
    axes[i].set_title(col)

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribusi Fitur Numerik")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "eda_numeric_distribution.png", dpi=150)
plt.show()


Kolom numerik: ['ips_1', 'ips_2', 'ips_3', 'ips_4', 'ips_5', 'ips_6', 'ips_7', 'jumlah_sks', 'umur', 'jumlah_tanggungan']


In [8]:
plt.figure(figsize=(12, 8))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Korelasi Fitur Numerik")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "eda_correlation_matrix.png", dpi=150)
plt.show()


In [9]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_feature_cols = [col for col in cat_cols if col != "status_kelulusan"]
print(f"Kolom kategorikal: {cat_feature_cols}")

if cat_feature_cols:
    fig, axes = plt.subplots(1, len(cat_feature_cols), figsize=(5 * len(cat_feature_cols), 4))
    if len(cat_feature_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, cat_feature_cols):
        df[col].value_counts().plot(kind="bar", ax=ax, color="#9b59b6", edgecolor="black")
        ax.set_title(col)
        ax.tick_params(axis="x", rotation=45)

    plt.suptitle("Distribusi Fitur Kategorikal")
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "eda_categorical_distribution.png", dpi=150)
    plt.show()


Kolom kategorikal: ['jenis_kelamin', 'asal_sekolah', 'organisasi', 'beasiswa', 'penghasilan_ortu']


In [10]:
ips_cols = [col for col in df.columns if col.startswith("ips_")]
print(f"Kolom IPS: {ips_cols}")

if ips_cols:
    fig, axes = plt.subplots(1, len(ips_cols), figsize=(3 * len(ips_cols), 4))
    if len(ips_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, ips_cols):
        df.boxplot(column=col, by="status_kelulusan", ax=ax)
        ax.set_title(col)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=45)

    plt.suptitle("IPS per Semester vs Status Kelulusan")
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "eda_ips_vs_target.png", dpi=150)
    plt.show()


Kolom IPS: ['ips_1', 'ips_2', 'ips_3', 'ips_4', 'ips_5', 'ips_6', 'ips_7']


# Data Preprocessing


In [11]:
df_processed = df.copy()

print("Missing values sebelum preprocessing:")
print(df_processed.isnull().sum()[df_processed.isnull().sum() > 0])

numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_processed.select_dtypes(include=["object"]).columns.tolist()

for col in numeric_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col] = df_processed[col].fillna(df_processed[col].median())

for col in cat_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col] = df_processed[col].fillna(df_processed[col].mode()[0])

print("Total missing values sesudah preprocessing:", df_processed.isnull().sum().sum())


Missing values sebelum preprocessing:
ips_3               20
umur                15
penghasilan_ortu    15
dtype: int64
Total missing values sesudah preprocessing: 0


In [12]:
ips_cols = [col for col in df_processed.columns if col.startswith("ips_")]

if ips_cols:
    df_processed["ipk_rata_rata"] = df_processed[ips_cols].mean(axis=1).round(2)

if len(ips_cols) >= 2:
    df_processed["ips_trend"] = (df_processed[ips_cols[-1]] - df_processed[ips_cols[0]]).round(2)
    df_processed["ips_std"] = df_processed[ips_cols].std(axis=1).round(4)

if "jumlah_sks" in df_processed.columns:
    df_processed["sks_ratio"] = (df_processed["jumlah_sks"] / 144).round(4)

print("Fitur baru yang dibuat:")
print([col for col in ["ipk_rata_rata", "ips_trend", "ips_std", "sks_ratio"] if col in df_processed.columns])
print(f"Shape setelah feature engineering: {df_processed.shape}")


Fitur baru yang dibuat:
['ipk_rata_rata', 'ips_trend', 'ips_std', 'sks_ratio']
Shape setelah feature engineering: (1000, 20)


In [13]:
target_col = "status_kelulusan"

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_processed[target_col])
X = df_processed.drop(columns=[target_col]).copy()

print("Mapping target:")
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

cat_feature_cols = X.select_dtypes(include=["object"]).columns.tolist()
print(f"Kolom kategorikal untuk encoding: {cat_feature_cols}")

if "penghasilan_ortu" in X.columns:
    ortu_order = [["<2jt", "2-5jt", "5-10jt", ">10jt"]]
    ordinal_encoder = OrdinalEncoder(categories=ortu_order, handle_unknown="use_encoded_value", unknown_value=-1)
    X["penghasilan_ortu"] = ordinal_encoder.fit_transform(X[["penghasilan_ortu"]])

nominal_cols = [col for col in cat_feature_cols if col != "penghasilan_ortu"]
if nominal_cols:
    X = pd.get_dummies(X, columns=nominal_cols, drop_first=True, dtype=int)

print(f"Jumlah fitur setelah encoding: {X.shape[1]}")
X.head()


Mapping target:
{'Drop Out': np.int64(0), 'Lulus Tepat Waktu': np.int64(1), 'Lulus Terlambat': np.int64(2)}
Kolom kategorikal untuk encoding: ['jenis_kelamin', 'asal_sekolah', 'organisasi', 'beasiswa', 'penghasilan_ortu']
Jumlah fitur setelah encoding: 20


,ips_1,ips_2,ips_3,ips_4,ips_5,ips_6,ips_7,jumlah_sks,umur,jumlah_tanggungan,penghasilan_ortu,ipk_rata_rata,ips_trend,ips_std,sks_ratio,jenis_kelamin_Perempuan,asal_sekolah_SMA,asal_sekolah_SMK,organisasi_Ya,beasiswa_Ya
0,2.12,1.56,1.79,3.02,2.72,2.18,2.94,150,24.0,3,2.0,2.33,0.82,0.5703,1.0417,1,0,1,1,0
1,3.85,2.63,1.74,3.39,3.42,2.42,1.52,116,28.0,4,3.0,2.71,-2.33,0.8863,0.8056,0,0,1,1,1
2,3.20,3.62,3.72,1.75,3.28,3.56,3.62,86,21.0,2,1.0,3.25,0.42,0.6885,0.5972,0,1,0,0,1
3,2.80,3.20,1.75,2.87,1.46,2.02,2.84,134,18.0,5,3.0,2.42,0.04,0.6661,0.9306,1,1,0,0,0
4,1.47,3.42,1.82,2.72,1.45,3.61,1.47,64,25.0,2,1.0,2.28,0.00,0.9552,0.4444,0,0,1,1,0


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")


X_train: (800, 20)
X_test : (200, 20)
y_train: (800,)
y_test : (200,)


In [15]:
scaler = StandardScaler()
feature_names = X_train.columns.tolist()

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names, index=X_test.index)

print("Scaling selesai menggunakan StandardScaler")
print(f"Jumlah fitur final: {len(feature_names)}")


Scaling selesai menggunakan StandardScaler
Jumlah fitur final: 20


In [16]:
PREPROCESSING_DIR = Path("preprocessing")
PREPROCESSING_DIR.mkdir(exist_ok=True)

X_train_scaled.to_csv(PREPROCESSING_DIR / "X_train.csv", index=False)
X_test_scaled.to_csv(PREPROCESSING_DIR / "X_test.csv", index=False)
pd.DataFrame(y_train, columns=["target"]).to_csv(PREPROCESSING_DIR / "y_train.csv", index=False)
pd.DataFrame(y_test, columns=["target"]).to_csv(PREPROCESSING_DIR / "y_test.csv", index=False)

joblib.dump(scaler, PREPROCESSING_DIR / "scaler.joblib")
joblib.dump(label_encoder, PREPROCESSING_DIR / "label_encoder.joblib")

feature_info = {
    "feature_names": feature_names,
    "n_features": len(feature_names),
    "n_train_samples": int(X_train_scaled.shape[0]),
    "n_test_samples": int(X_test_scaled.shape[0]),
    "target_classes": label_encoder.classes_.tolist(),
}

with open(PREPROCESSING_DIR / "feature_info.json", "w", encoding="utf-8") as f:
    json.dump(feature_info, f, indent=2)

print("Data preprocessing selesai. File data siap latih tersimpan di folder preprocessing/:")
for path in sorted(PREPROCESSING_DIR.iterdir()):
    print(f"- {path.name}")


Data preprocessing selesai. File data siap latih tersimpan di folder preprocessing/:
- feature_info.json
- label_encoder.joblib
- scaler.joblib
- X_test.csv
- X_train.csv
- y_test.csv
- y_train.csv
